# Machine Translation: Fine-tuning English → Hindi

A simple, end-to-end example of fine-tuning a sequence-to-sequence model to translate English into Hindi.

> Note: translation is a **seq2seq** task, so we use an encoder-decoder model (`Helsinki-NLP/opus-mt-en-hi`) with `Seq2SeqTrainer` — BERT (encoder-only) can't generate text. The workflow otherwise mirrors the classification notebooks.

**Pipeline:**
1. Download a public dataset (IIT Bombay English-Hindi parallel corpus)
2. Save it to CSV files (`train.csv` / `test.csv`)
3. Read the data back from the CSV files
4. Tokenize and fine-tune the model
5. Evaluate (BLEU) and run inference
6. Save and reload the model

Built with `transformers` + `torch`.

## 0. Install dependencies

Run once if these aren't already installed. `sentencepiece` is required by the Marian tokenizer; `sacrebleu` is the standard translation metric.

In [ ]:
!pip install torch transformers datasets sentencepiece sacrebleu pandas

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import sacrebleu
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## 2. Download a public dataset and 3. save it to CSV

We use the **IIT Bombay English-Hindi** corpus (`cfilt/iitb-english-hindi`). Each example is a `translation` dict with `en` and `hi` fields.

We flatten it to two flat columns (`en`, `hi`) — one sentence pair per row — and take a small subset so it trains quickly.

In [ ]:
# Load from the Hugging Face Hub
raw = load_dataset("cfilt/iitb-english-hindi")

# Take a small subset so it trains quickly (remove .select(...) to use the full data)
train_subset = raw["train"].shuffle(seed=42).select(range(3000))
test_subset = raw["validation"].shuffle(seed=42).select(range(500))


def to_csv(split, path):
    df = pd.DataFrame(
        {
            "en": [ex["en"] for ex in split["translation"]],
            "hi": [ex["hi"] for ex in split["translation"]],
        }
    )
    df.to_csv(path, index=False)


to_csv(train_subset, "train.csv")
to_csv(test_subset, "test.csv")
print("Saved train.csv and test.csv")

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Saved train.csv and test.csv


## 4. Read the data back from the CSV files

In [ ]:
train_df = pd.read_csv("train.csv").dropna()
test_df = pd.read_csv("test.csv").dropna()

print(train_df.shape, test_df.shape)
train_df.head()

(2986, 2) (500, 2)


,en,hi
0,on the intuition.,अंतर्ज्ञान पर।
1,Ceiba pentandra,कण्टकारी
2,Have you not regarded how your Lord dealt with...,क्या तुमने देखा नहीं कि तुम्हारे आद के साथ क्य...
3,Provided that no such order shall be made afte...,"ऐसी कोई आदेश अक्टूबर, 1991 के 1 दिन से दो वर्ष..."
4,File format is unknown or unsupported,फ़ाइल प्रारूप अज्ञात या असमर्थित है


## 5. Tokenize

For seq2seq we tokenize the English side as the input and the Hindi side as the target (`text_target`). We wrap the encodings in a tiny `torch.utils.data.Dataset`.

In [ ]:
MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, source, target, tokenizer, max_len=MAX_LEN):
        self.enc = tokenizer(
            list(source),
            text_target=list(target),
            truncation=True,
            max_length=max_len,
        )

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels": self.enc["labels"][idx],
        }


train_ds = TranslationDataset(train_df["en"], train_df["hi"], tokenizer)
test_ds = TranslationDataset(test_df["en"], test_df["hi"], tokenizer)

# Dynamically pads inputs and labels (with -100) to the longest item in each batch
data_collator = DataCollatorForSeq2Seq(tokenizer, model=None)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


## 6. Fine-tune (training)

We compute **BLEU** on the generated translations. `predict_with_generate=True` tells the trainer to run actual decoding during evaluation.

In [ ]:
'''
ENG -> Hi How are you ?
HI -> Namaste aap kaise hai ?

MODEL PRED -> Namaste tum kaise ho ?


'''

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator.model = model


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in the labels so they can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # sacrebleu.corpus_bleu expects: hypotheses list + list of reference lists
    result = sacrebleu.corpus_bleu(
        [p.strip() for p in decoded_preds],
        [[l.strip() for l in decoded_labels]],
    )
    return {"bleu": result.score}


training_args = Seq2SeqTrainingArguments(
    output_dir="./mt-output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Bleu
1,3.200909,4.067471,8.184533
2,2.940980,4.048059,8.458254
3,2.735521,4.045396,8.251281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=561, training_loss=3.1379675023696003, metrics={'train_runtime': 189.8014, 'train_samples_per_second': 47.197, 'train_steps_per_second': 2.956, 'total_flos': 142365717430272.0, 'train_loss': 3.1379675023696003, 'epoch': 3.0})

## 7. Evaluate

In [ ]:
metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch,Bleu
2.735521,4.048059,3,8.458254


{'eval_loss': 4.04805850982666, 'eval_bleu': 8.458253812085882}


## 8. Inference on new text

In [ ]:
def translate(texts, max_len=MAX_LEN):
    model.eval()
    enc = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len,
    ).to(model.device)
    with torch.no_grad():
        generated = model.generate(**enc, max_length=max_len, num_beams=4)
    return tokenizer.batch_decode(generated, skip_special_tokens=True)


samples = [
    "Hello, how are you?",
    "India is a beautiful country.",
    "I am learning machine translation.",
    "What time is the meeting tomorrow?",
]

for en, hi in zip(samples, translate(samples)):
    print(f"EN: {en}\nHI: {hi}\n")

EN: Hello, how are you?
HI: हैलो, तुम कैसे हो?

EN: India is a beautiful country.
HI: भारत एक सुंदर देश है।

EN: I am learning machine translation.
HI: मैं मशीन अनुवाद सीख रहा हूँ।

EN: What time is the meeting tomorrow?
HI: क्या समय है बैठक कल है?



## 9. Save the fine-tuned model

In [ ]:
trainer.save_model("./mt-en-hi")
tokenizer.save_pretrained("./mt-en-hi")
print("Model saved to ./mt-en-hi")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./mt-en-hi
